# Data Preparation: Balancing the Dataset

## Goal
Create a balanced dataset for the "Big 5" classes with approximately **200 samples per class**.

### Strategy
1.  **Undersampling**: Majority classes (Siren, Car Horn, Dog Bark from US8K) will be limited to ~200 samples.
2.  **Augmentation**: Minority classes (Crying Baby, Door Knock from ESC-50) will be augmented (Time Stretch, Pitch Shift, Noise Injection) to reach ~200 samples.
3.  **Standardization**: All files will be saved to `data/processed/audio` and a `master_metadata.csv` will be created.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import numpy as np
import os
import shutil
from tqdm import tqdm

# Settings
TARGET_COUNT = 200
RANDOM_SEED = 42
PROCESSED_AUDIO_DIR = '../data/processed/audio/'
PROCESSED_META_DIR = '../data/processed/'

# Ensure directories exist
os.makedirs(PROCESSED_AUDIO_DIR, exist_ok=True)

# Plot style
plt.style.use('ggplot')

## 1. Load and Filter Data (From Previous Step)

In [ ]:
# Define Targets
ESC50_TARGETS = {
    'siren': ('Siren', 4),
    'car_horn': ('Car Horn', 3),
    'crying_baby': ('Crying Baby', 3),
    'door_wood_knock': ('Door Knock', 2),
    'dog': ('Dog Bark', 2)
}

US8K_TARGETS = {
    'siren': ('Siren', 4),
    'car_horn': ('Car Horn', 3),
    'dog_bark': ('Dog Bark', 2)
}

# Load ESC-50
esc50_path = '../data/raw/ESC-50-master/meta/esc50.csv'
esc50_audio_dir = '../data/raw/ESC-50-master/audio/'
df_esc50 = pd.read_csv(esc50_path)
df_esc50_selected = df_esc50[df_esc50['category'].isin(ESC50_TARGETS.keys())].copy()
df_esc50_selected['target_class'] = df_esc50_selected['category'].apply(lambda x: ESC50_TARGETS[x][0])
df_esc50_selected['urgency'] = df_esc50_selected['category'].apply(lambda x: ESC50_TARGETS[x][1])
df_esc50_selected['source'] = 'ESC-50'
df_esc50_selected['full_path'] = df_esc50_selected['filename'].apply(lambda x: os.path.join(esc50_audio_dir, x))

# Load UrbanSound8K
us8k_path = '../data/raw/archive/UrbanSound8K.csv'
us8k_audio_dir = '../data/raw/archive/'
df_us8k = pd.read_csv(us8k_path)
df_us8k_selected = df_us8k[df_us8k['class'].isin(US8K_TARGETS.keys())].copy()
df_us8k_selected['target_class'] = df_us8k_selected['class'].apply(lambda x: US8K_TARGETS[x][0])
df_us8k_selected['urgency'] = df_us8k_selected['class'].apply(lambda x: US8K_TARGETS[x][1])
df_us8k_selected['source'] = 'UrbanSound8K'
df_us8k_selected['full_path'] = df_us8k_selected.apply(
    lambda row: os.path.join(us8k_audio_dir, f"fold{row.fold}", row.slice_file_name), axis=1
)

# Combine
df_combined = pd.concat([
    df_esc50_selected[['filename', 'target_class', 'urgency', 'source', 'full_path']],
    df_us8k_selected.rename(columns={'slice_file_name': 'filename'})[['filename', 'target_class', 'urgency', 'source', 'full_path']]
], ignore_index=True)

print("Original Distribution:")
print(df_combined['target_class'].value_counts())

## 2. Undersampling (Majority Classes)

In [ ]:
balanced_data = []

for class_name in df_combined['target_class'].unique():
    class_subset = df_combined[df_combined['target_class'] == class_name]
    count = len(class_subset)
    
    if count > TARGET_COUNT:
        # Undersample
        print(f"Undersampling {class_name}: {count} -> {TARGET_COUNT}")
        sampled_subset = class_subset.sample(n=TARGET_COUNT, random_state=RANDOM_SEED)
        balanced_data.append(sampled_subset)
    else:
        # Keep as is (will augment later)
        print(f"Keeping {class_name}: {count} (will augment)")
        balanced_data.append(class_subset)

df_balanced_initial = pd.concat(balanced_data, ignore_index=True)
print("\nAfter Undersampling:")
print(df_balanced_initial['target_class'].value_counts())

## 3. Augmentation Logic
Defining functions to create variations of audio files.

In [ ]:
def augment_audio(y, sr):
    """Generates 3 variations of the input audio."""
    variations = []
    
    # 1. Time Stretch (Faster)
    try:
        y_fast = librosa.effects.time_stretch(y, rate=1.2)
        variations.append(('stretch_fast', y_fast))
    except:
        pass
        
    # 2. Pitch Shift (Lower)
    try:
        y_low = librosa.effects.pitch_shift(y, sr=sr, n_steps=-2)
        variations.append(('pitch_low', y_low))
    except:
        pass
        
    # 3. Noise Injection
    try:
        noise = np.random.randn(len(y))
        y_noise = y + 0.005 * noise
        variations.append(('noise_inject', y_noise))
    except:
        pass
    
    return variations

## 4. Processing and Saving Audio
- Save 'original' files to `processed/audio`.
- If class count < TARGET_COUNT, generate augmentations and save them too.

In [ ]:
final_metadata = []

print("Processing Audio Files...")

for class_name in df_balanced_initial['target_class'].unique():
    class_subset = df_balanced_initial[df_balanced_initial['target_class'] == class_name]
    original_count = len(class_subset)
    needed = TARGET_COUNT - original_count
    
    # If we need augmentation (e.g., Crying Baby has 40, need 160 more)
    augment_factor = 0
    if needed > 0:
        # Calculate how many augmentations per file
        augment_factor = int(np.ceil(needed / original_count))
        print(f"Augmenting {class_name} (factor: {augment_factor}x) to reach target.")
    
    for idx, row in tqdm(class_subset.iterrows(), total=original_count, desc=class_name):
        src_path = row['full_path']
        fname = row['filename']
        target_class = row['target_class']
        urgency = row['urgency']
        
        # Load Audio
        try:
            y, sr = librosa.load(src_path, sr=None) # Keep original SR
            
            # 1. Save Original
            dst_filename = f"original_{fname}"
            dst_path = os.path.join(PROCESSED_AUDIO_DIR, dst_filename)
            
            sf.write(dst_path, y, sr)
            
            final_metadata.append({
                'filename': dst_filename,
                'filepath': dst_path,
                'class': target_class,
                'urgency': urgency,
                'type': 'original',
                'source': row['source'],
                'source_file': fname
            })
            
            # 2. Augment if needed
            if needed > 0:
                variations = augment_audio(y, sr)
                # Take only as many as needed to avoid overshooting too much 
                # (simple logic: take all generate, we filter excess later if strictly needed, 
                # but here having slightly > 200 is fine)
                
                for var_name, y_aug in variations[:augment_factor]:
                    aug_filename = f"aug_{var_name}_{fname}"
                    aug_path = os.path.join(PROCESSED_AUDIO_DIR, aug_filename)
                    
                    sf.write(aug_path, y_aug, sr)
                    
                    final_metadata.append({
                        'filename': aug_filename,
                        'filepath': aug_path,
                        'class': target_class,
                        'urgency': urgency,
                        'type': 'augmented',
                        'source': row['source'],
                        'source_file': fname
                    })
                    
        except Exception as e:
            print(f"Error processing {src_path}: {e}")

## 5. Save Master Metadata
Create the final CSV that will be used for training.

In [ ]:
df_final = pd.DataFrame(final_metadata)

# Ensure roughly 200 per class (Augmentation potentially created slightly more)
# We can optionally trim here to be exactly 200, but keeping all valid augments is fine.

print("Final Dataset Distribution:")
print(df_final['class'].value_counts())

# Save
save_path = os.path.join(PROCESSED_META_DIR, 'master_metadata.csv')
df_final.to_csv(save_path, index=False)
print(f"Saved master metadata to {save_path}")

In [ ]:
# Final Visualization
plt.figure(figsize=(10, 6))
df_final['class'].value_counts().plot(kind='bar', color='green')
plt.title('Final Balanced Dataset Counts')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()